# Phase 2 Finalization — Cellwise Tests, Effect CIs, Condition and Task Contrasts

This supplement corrects the global-complete-case issue by constructing PRE–POST pairs at each participant × task × condition × band × ROI cell. It recomputes primary tests, adds confidence intervals, direct condition-change contrasts, and matched Math-vs-Reading contrasts. Holm correction is applied within the stated families.

**CI limitation:** Mean-change intervals are t-based. Rank-effect intervals use an approximate Fisher-z transform and are not bootstrap CIs; interpret them as approximate. This supplement is an inferential extension, not a replacement for the original extraction/QC steps.

In [6]:
import pandas as pd, numpy as np, json
from pathlib import Path
from scipy.stats import wilcoxon, mannwhitneyu, rankdata, t
from statsmodels.stats.multitest import multipletests

# Locate the project data folder portably (macOS/Windows/Linux/Jupyter).
def find_input_root(filename="eeg_roi_power_long.csv"):
    candidates = [Path.cwd(), *Path.cwd().parents]
    # Common project location on macOS, if notebook is opened outside the analysis folder.
    candidates += [Path.home() / "Documents" / "Term Project" / "analysis",
                   Path.home() / "Documents" / "Term Project",
                   Path("/mnt/data")]  # Colab/container fallback
    for candidate in candidates:
        candidate = candidate.expanduser()
        if (candidate / filename).is_file():
            return candidate.resolve()
    # Last-resort search in the user's Documents folder and current project tree.
    search_roots = [Path.cwd(), Path.home() / "Documents"]
    for search_root in search_roots:
        if search_root.exists():
            try:
                found = next(search_root.rglob(filename), None)
                if found:
                    return found.parent.resolve()
            except (PermissionError, OSError):
                pass
    raise FileNotFoundError(
        f"Could not locate {filename}. Place it in the notebook folder or "
        "set ROOT manually to the folder containing it."
    )

ROOT = find_input_root()
OUT = ROOT / "eeg_phase2_review_fixed_outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("Input directory:", ROOT)
print("Output directory:", OUT)

print("Working directory:", ROOT)
print("Output directory:", OUT)
d=pd.read_csv(ROOT/'eeg_roi_power_long.csv'); d['log10_absolute']=np.log10(d.absolute_power_roi.astype(float))
metrics={'relative_power':'relative_power_roi','log10_absolute':'log10_absolute'}; base=['subject_id','group','task','condition','band','roi']; rev={v:k for k,v in metrics.items()}
w=d.pivot_table(index=base,columns='session',values=list(metrics.values()),aggfunc='first');w.columns=[f'{rev[a]}_{b}' for a,b in w.columns];w=w.reset_index()
for m in metrics:w[f'{m}_change']=w[f'{m}_POST']-w[f'{m}_PRE']

def p_w(x,y):
 x=np.asarray(x,float);y=np.asarray(y,float); ok=np.isfinite(x)&np.isfinite(y);x=x[ok];y=y[ok]
 if len(x)<2 or np.allclose(x-y,0):return np.nan,np.nan
 try:r=wilcoxon(y,x,alternative='two-sided',zero_method='wilcox',method='auto');return float(r.statistic),float(r.pvalue)
 except:return np.nan,np.nan

def paired_rb(x,y):
 diff=np.asarray(y)-np.asarray(x);diff=diff[np.isfinite(diff)& (diff!=0)]
 if len(diff)==0:return np.nan
 ranks=rankdata(np.abs(diff));return float((ranks[diff>0].sum()-ranks[diff<0].sum())/ranks.sum())

def mw(a,b):
 a=np.asarray(a,float);b=np.asarray(b,float);a=a[np.isfinite(a)];b=b[np.isfinite(b)]
 if len(a)<2 or len(b)<2:return np.nan,np.nan,np.nan
 try:r=mannwhitneyu(a,b,alternative='two-sided',method='auto');return float(r.statistic),float(r.pvalue),float(2*r.statistic/(len(a)*len(b))-1)
 except Exception:
  return np.nan,np.nan,np.nan

def mean_ci(x):
 x=np.asarray(x,float);x=x[np.isfinite(x)];n=len(x);mean=np.mean(x) if n else np.nan
 if n<2:return mean,np.nan,np.nan
 se=np.std(x,ddof=1)/np.sqrt(n);crit=t.ppf(.975,n-1);return mean,mean-crit*se,mean+crit*se

def effect_ci(e,n):
 if not np.isfinite(e) or n<4:return np.nan,np.nan
 # Fisher-z approximation for rank correlation effect, acknowledged approximate
 z=np.arctanh(np.clip(e,-.999,.999));se=1/np.sqrt(n-3)
 return np.tanh(z-1.96*se),np.tanh(z+1.96*se)

within=[];between=[]
for metric in metrics:
 pre,post,ch=f'{metric}_PRE',f'{metric}_POST',f'{metric}_change'
 for (task,cond,band,roi),cell in w.groupby(['task','condition','band','roi']):
  for group in ['Control','Experiment']:
   g=cell[cell.group==group].dropna(subset=[pre,post]); n=len(g);stat,p=p_w(g[pre],g[post]);rb=paired_rb(g[pre],g[post]);m,lo,hi=mean_ci(g[ch]);rblo,rbhi=effect_ci(rb,n)
   within.append(dict(metric=metric,task=task,condition=cond,band=band,roi=roi,group=group,n_pairs=n,statistic=stat,p_raw=p,paired_rank_biserial=rb,rank_effect_CI95_low_approx=rblo,rank_effect_CI95_high_approx=rbhi,mean_change=m,mean_change_CI95_low=lo,mean_change_CI95_high=hi,median_change=g[ch].median()))
  a=cell[cell.group=='Experiment'].dropna(subset=[ch])[ch];b=cell[cell.group=='Control'].dropna(subset=[ch])[ch];u,p,rb=mw(a,b);lo,hi=effect_ci(rb,len(a)+len(b))
  ma,malo,mahi=mean_ci(a);mb,blo,bhi=mean_ci(b)
  between.append(dict(metric=metric,task=task,condition=cond,band=band,roi=roi,n_experiment=len(a),n_control=len(b),U=u,p_raw=p,rank_biserial_experiment_vs_control=rb,rank_effect_CI95_low_approx=lo,rank_effect_CI95_high_approx=hi,mean_change_experiment=ma,mean_change_experiment_CI95_low=malo,mean_change_experiment_CI95_high=mahi,mean_change_control=mb,mean_change_control_CI95_low=blo,mean_change_control_CI95_high=bhi))
within=pd.DataFrame(within);between=pd.DataFrame(between)
within['p_holm']=np.nan
for _,ix in within.groupby(['task','metric','group']).groups.items():
 pv=within.loc[ix,'p_raw'];ok=pv.notna();
 if ok.any():within.loc[pv.index[ok],'p_holm']=multipletests(pv[ok],method='holm')[1]
between['p_holm']=np.nan
for _,ix in between.groupby(['task','metric']).groups.items():
 pv=between.loc[ix,'p_raw'];ok=pv.notna();
 if ok.any():between.loc[pv.index[ok],'p_holm']=multipletests(pv[ok],method='holm')[1]
within.to_csv(OUT/'phase2_cellwise_primary_tests_effect_CI_within.csv',index=False);between.to_csv(OUT/'phase2_cellwise_primary_tests_effect_CI_between.csv',index=False)
# Direct condition contrasts on individual PRE-POST change scores, using participants complete for the pair.
condition_specs={'Math':[('Activity1','Baseline'),('Activity2','Baseline'),('Activity2','Activity1')],'Reading':[('Activity','Baseline')]}
cond_w=[];cond_b=[]
for metric in metrics:
 for task,specs in condition_specs.items():
  sub=w[w.task==task]
  for band in sorted(sub.band.unique()):
   for roi in sorted(sub.roi.unique()):
    for ca,cb in specs:
     s=sub[(sub.band==band)&(sub.roi==roi)&sub.condition.isin([ca,cb])]
     pv=s.pivot_table(index=['subject_id','group'],columns='condition',values=f'{metric}_change',aggfunc='first').dropna(subset=[ca,cb]).reset_index()
     pv['contrast']=pv[ca]-pv[cb]
     for grp in ['Control','Experiment']:
      g=pv[pv.group==grp];stat,p=p_w(g[cb],g[ca]);rb=paired_rb(g[cb],g[ca]);m,lo,hi=mean_ci(g.contrast)
      cond_w.append(dict(metric=metric,task=task,band=band,roi=roi,group=grp,contrast=f'{ca} minus {cb}',n_pairs=len(g),statistic=stat,p_raw=p,paired_rank_biserial=rb,mean_contrast=m,mean_contrast_CI95_low=lo,mean_contrast_CI95_high=hi))
     a=pv[pv.group=='Experiment'].contrast;b=pv[pv.group=='Control'].contrast;u,p,rb=mw(a,b);lo,hi=effect_ci(rb,len(a)+len(b))
     cond_b.append(dict(metric=metric,task=task,band=band,roi=roi,contrast=f'{ca} minus {cb}',n_experiment=len(a),n_control=len(b),U=u,p_raw=p,rank_biserial=rb,rank_effect_CI95_low_approx=lo,rank_effect_CI95_high_approx=hi))
cw=pd.DataFrame(cond_w);cb=pd.DataFrame(cond_b)
cw['p_holm']=np.nan
for _,ix in cw.groupby(['task','metric','group','contrast']).groups.items():
 pv=cw.loc[ix,'p_raw'];ok=pv.notna();
 if ok.any():cw.loc[pv.index[ok],'p_holm']=multipletests(pv[ok],method='holm')[1]
cb['p_holm']=np.nan
for _,ix in cb.groupby(['task','metric','contrast']).groups.items():
 pv=cb.loc[ix,'p_raw'];ok=pv.notna();
 if ok.any():cb.loc[pv.index[ok],'p_holm']=multipletests(pv[ok],method='holm')[1]
cw.to_csv(OUT/'phase2_direct_condition_contrasts_within_group.csv',index=False);cb.to_csv(OUT/'phase2_direct_condition_contrasts_between_group.csv',index=False)
# Matched task contrasts: baseline vs baseline, and math activity mean(A1,A2) vs reading activity.
taskw=[];taskb=[]
for metric in metrics:
 for band in sorted(w.band.unique()):
  for roi in sorted(w.roi.unique()):
   mbase=w[(w.task=='Math')&(w.condition=='Baseline')&(w.band==band)&(w.roi==roi)][['subject_id','group',f'{metric}_change']]
   rbase=w[(w.task=='Reading')&(w.condition=='Baseline')&(w.band==band)&(w.roi==roi)][['subject_id','group',f'{metric}_change']]
   mact=w[(w.task=='Math')&w.condition.isin(['Activity1','Activity2'])&(w.band==band)&(w.roi==roi)].groupby(['subject_id','group'])[f'{metric}_change'].mean().reset_index()
   ract=w[(w.task=='Reading')&(w.condition=='Activity')&(w.band==band)&(w.roi==roi)][['subject_id','group',f'{metric}_change']]
   for label,ma,ra in [('Baseline',mbase,rbase),('Activity',mact,ract)]:
    merged=ma.merge(ra,on=['subject_id','group'],suffixes=('_math','_reading')).dropna()
    for grp in ['Control','Experiment']:
     g=merged[merged.group==grp];x=g[f'{metric}_change_reading'];y=g[f'{metric}_change_math'];stat,p=p_w(x,y);rb=paired_rb(x,y);m,lo,hi=mean_ci(y-x)
     taskw.append(dict(metric=metric,task='Math vs Reading',band=band,roi=roi,comparison=label,group=grp,n_matched=len(g),statistic=stat,p_raw=p,paired_rank_biserial=rb,mean_math_minus_reading_change=m,CI95_low=lo,CI95_high=hi))
    ex=merged[merged.group=='Experiment'];co=merged[merged.group=='Control'];ed=ex[f'{metric}_change_math']-ex[f'{metric}_change_reading'];cd=co[f'{metric}_change_math']-co[f'{metric}_change_reading'];u,p,rb=mw(ed,cd);lo,hi=effect_ci(rb,len(ed)+len(cd))
    taskb.append(dict(metric=metric,band=band,roi=roi,comparison=label,n_experiment=len(ed),n_control=len(cd),U=u,p_raw=p,rank_biserial_experiment_vs_control_task_difference=rb,rank_effect_CI95_low_approx=lo,rank_effect_CI95_high_approx=hi))
tw=pd.DataFrame(taskw);tb=pd.DataFrame(taskb)
tw['p_holm']=np.nan
for _,ix in tw.groupby(['metric','comparison','group']).groups.items():
 pv=tw.loc[ix,'p_raw'];ok=pv.notna();
 if ok.any():tw.loc[pv.index[ok],'p_holm']=multipletests(pv[ok],method='holm')[1]
tb['p_holm']=np.nan
for _,ix in tb.groupby(['metric','comparison']).groups.items():
 pv=tb.loc[ix,'p_raw'];ok=pv.notna();
 if ok.any():tb.loc[pv.index[ok],'p_holm']=multipletests(pv[ok],method='holm')[1]
tw.to_csv(OUT/'phase2_matched_task_contrasts_within_group.csv',index=False);tb.to_csv(OUT/'phase2_matched_task_difference_between_groups.csv',index=False)
summary={
 'input_rows':len(d),'participants':int(d.subject_id.nunique()),'cellwise_primary_within_tests':len(within),'cellwise_primary_between_tests':len(between),
 'cellwise_primary_within_holm_significant':int((within.p_holm<.05).sum()),'cellwise_primary_between_holm_significant':int((between.p_holm<.05).sum()),
 'condition_contrasts_within_tests':len(cw),'condition_contrasts_between_tests':len(cb),'condition_contrasts_within_holm_sig':int((cw.p_holm<.05).sum()),'condition_contrasts_between_holm_sig':int((cb.p_holm<.05).sum()),
 'task_contrasts_within_tests':len(tw),'task_difference_between_tests':len(tb),'task_contrasts_within_holm_sig':int((tw.p_holm<.05).sum()),'task_difference_between_holm_sig':int((tb.p_holm<.05).sum()),
 'pairing_method':'cellwise PRE-POST complete pairs (not global complete-case across every metric/condition)',
 'CI_method':'t-based 95% CI for mean change; approximate Fisher-z CI for rank effect sizes. CIs not bootstrap.'}
json.dump(summary,open(OUT/'phase2_finalization_audit_summary.json','w'),indent=2)
print(json.dumps(summary,indent=2));print('primary significant within',within[within.p_holm<.05][['task','metric','condition','band','roi','group','p_raw','p_holm']].to_string(index=False));print('primary significant between',between[between.p_holm<.05][['task','metric','condition','band','roi','p_raw','p_holm']].to_string(index=False));print('condition significant within',cw[cw.p_holm<.05][['task','metric','band','roi','group','contrast','p_raw','p_holm']].to_string(index=False));print('condition significant between',cb[cb.p_holm<.05][['task','metric','band','roi','contrast','p_raw','p_holm']].to_string(index=False));print('task significant within',tw[tw.p_holm<.05][['metric','band','roi','comparison','group','p_raw','p_holm']].to_string(index=False));print('task significant between',tb[tb.p_holm<.05][['metric','band','roi','comparison','p_raw','p_holm']].to_string(index=False))


Input directory: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs
Output directory: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs/eeg_phase2_review_fixed_outputs
Working directory: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs
Output directory: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs/eeg_phase2_review_fixed_outputs
{
  "input_rows": 8000,
  "participants": 63,
  "cellwise_primary_within_tests": 500,
  "cellwise_primary_between_tests": 250,
  "cellwise_primary_within_holm_significant": 0,
  "cellwise_primary_between_holm_significant": 0,
  "condition_contrasts_within_tests": 400,
  "condition_contrasts_between_tests": 200,
  "condition_contrasts_within_holm_sig": 0,
  "condition_contrasts_between_holm_sig": 0,
  "task_contrasts_within_tests": 200,
  "task_difference_between_tests": 100,
  "task_contrasts_within_holm_sig": 0,
  "task_difference_between_holm_sig": 0,
  "pairing_method": 

## Sensitivity analysis: flagged absolute-power observations removed

This addendum recalculates cellwise PRE–POST and Experiment-vs-Control tests after removing the union of IQR and MAD flagged observations. This is exploratory robustness analysis only; flags do not identify artifacts automatically.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Automatically locate the input file
filename = "eeg_roi_power_long.csv"
search_locations = [Path.cwd(), Path.home() / "Term Project", Path.home()]

csv_path = None

for location in search_locations:
    if location.exists():
        matches = list(location.rglob(filename))
        if matches:
            csv_path = matches[0]
            break

if csv_path is None:
    raise FileNotFoundError(
        f"Could not locate {filename}. "
        "Place the CSV in your Term Project folder."
    )

# Use the directory containing the input data
root = csv_path.parent

# Create output directory safely
out = root / "eeg_phase2_statistics_outputs"
out.mkdir(parents=True, exist_ok=True)

print("Input file:", csv_path)
print("Output directory:", out)

# Load data
data = pd.read_csv(csv_path)
data["log10_absolute"] = np.log10(
    data["absolute_power_roi"].astype(float)
)

Input file: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs/eeg_roi_power_long.csv
Output directory: /Users/ananyosen/Documents/Term Project/analysis/eeg_roi_step2_outputs/eeg_phase2_statistics_outputs
